## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Patrick Lee

**Date:** 10/2/2026

### Q1

1. Regression predicts a numeric value such as the price of a car. Classification predicts a category or label such as the type of a car.

2. A confusion table compares the actual classes to the classes predicted by the model. It shows how many predictions were correct and which classes the model commonly confused with each other.

3. SSE measures the total squared difference between the actual values and the predicted values. A lower SSE means the model's predictions were closer to the actual values.

4. Overfitting happens when a model follows the training data too closely and does not predict new data well. Underfitting happens when the model is too simple to find important patterns in the data.

5. Splitting the data lets us fit the model on the training set and evaluate it on data that was not used to fit the model. Comparing accuracy or SSE on the validation set helps us choose a value of (k) that should perform well on new data. This prevents us from choosing a model just because it fits the training data well.

6. A class label is simple and shows which class the model predicts. However it does not show how certain the model is. A probability distribution shows the estimated probability for each class, which can help show when a prediction is uncertain. However, the probabilities are more difficult to interpret and may not accurately show how certain the model actually is.

In [42]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/land_mines.csv")

print(df[["voltage", "height", "soil"]].describe())
print(df["mine_type"].value_counts())
# The dataset has 338 observations and there are no missing values in the three features
# Voltage, height, and soil are numeric. Mine type is the target label
# The five mine types are fairly balanced. Each type has 65 to 71 observations

          voltage      height        soil
count  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550
std      0.195819    0.306043    0.344244
min      0.197734    0.000000    0.000000
25%      0.309737    0.272727    0.200000
50%      0.359516    0.545455    0.600000
75%      0.482628    0.727273    0.800000
max      0.999999    1.000000    1.000000
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64


In [43]:
from sklearn.model_selection import train_test_split

classification_X_raw = df[["voltage", "height", "soil"]]
classification_y = df["mine_type"]

(
    classification_X_train_raw,
    classification_X_valid_raw,
    classification_y_train,
    classification_y_valid,
) = train_test_split(
    classification_X_raw,
    classification_y,
    test_size=0.50,
    random_state=65,
)
# I split the data evenly. 50% used for training and 50% used for validation
# This is 169 observations in each set

In [44]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

classification_scaler = MinMaxScaler()
classification_X_train_scaled = classification_scaler.fit_transform(
    classification_X_train_raw
)
classification_X_valid_scaled = classification_scaler.transform(
    classification_X_valid_raw
)

classification_ks = np.arange(1, 51)
classification_valid_mistakes = []

for classification_k in classification_ks:
    classification_candidate = KNeighborsClassifier(
        n_neighbors=int(classification_k)
    )
    classification_candidate.fit(
        classification_X_train_scaled,
        classification_y_train,
    )
    classification_valid_hat = classification_candidate.predict(
        classification_X_valid_scaled
    )
    classification_valid_mistakes.append(
        len(classification_y_valid)
        - sum(
            classification_y_valid.to_numpy()
            == classification_valid_hat
        )
    )

classification_k_star = int(
    classification_ks[np.argmin(classification_valid_mistakes)]
)
print("Selected k:", classification_k_star)
# Tested values of (k) from 1 to 50 and counted the number of incorrect validation predictions for each one
# The fewest mistakes occurred when (k=2) which was used for the final model.
# I fitted the scaler on the training data and then used the same scaler on the validation data so that the validation data would not affect the model

Selected k: 2


In [45]:
classification_model = KNeighborsClassifier(
    n_neighbors=classification_k_star
).fit(
    classification_X_train_scaled,
    classification_y_train,
)

classification_valid_hat = classification_model.predict(
    classification_X_valid_scaled
)

print(pd.crosstab(classification_y_valid, classification_valid_hat))

classification_accuracy = (
    sum(
        classification_y_valid.to_numpy()
        == classification_valid_hat
    )
    / len(classification_y_valid)
)
print("Validation accuracy:", classification_accuracy)
# The model predicted 76/169 validation observations (45%)
# It performed best on mine type 2 (33/38), worst on mine type 5 (only 3/29)
# The confusion table shows that types 3, 4, and 5 were frequently confused with other mine types

col_0       1   2   3  4  5
mine_type                  
1          21   1   6  4  2
2           0  33   2  3  0
3           5   5  10  6  4
4          13   5  10  9  1
5          10   1   8  7  3
Validation accuracy: 0.44970414201183434


##### 2.5
I probably wouldn't use the model to decide how a suspected mine should actually be handled since its validation accuracy is only around 45% and the accuracy varies a lot across the different kinds of mines. I think for it to be a useful tools, someone could use it to initially screen any mine that is found, but to accurately predict, the human should confirm using other measurements and experts. The confusion table can be used as one of those measurements since it shows that some types of mines are a lot more likely to be falsely classified over others.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 4810ee370a86fc3195c31774277ce6f3974ee6e6
- **AI tool and model used:** Claude Sonnet 5.5

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

1. "I'm doing Phase 2 for A03. Look at my Phase 1 notebook (A03-knn/notebook.ipynb) and suggest revisions and improvements based on the assignment requirements."
2. "Give me the proposed changes as a concise numbered list, and give me the revised Phase 2 code so I can paste it in myself. Don't touch the frozen Phase 1 cells."


### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. Move the answers to Q2.3 (how k is selected) and Q2.4 (how accurate, where more or less accurate) out of code comments and into written markdown answers, since the assignment asks for explanations.
2. Rewrite the Q2.5 advice to address the prompt's hint directly: a type can have many correct predictions yet still produce many wrong ones. Use the confusion table to show this (e.g., predictions of type 1 are right only 21/49 times even though 21/34 true type 1 mines are caught).
3. Fix the Q1.5 wording: splitting the data improves our choice of k and gives an honest estimate of new-data performance, rather than improving the model itself.


### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1. Written answers for Q2.3 and Q2.4 | Accept | The assignment asks me to explain how I select k and where the model is more or less accurate, and Phase 1 only had code comments. I checked every number against the confusion table. |
| 2. Q2.5 tied to the error pattern | Accept | My Phase 1 answer was general. The question specifically points at accurate predictions that still come with many mistakes, and the confusion table's column totals show exactly that. I verified the totals by running the code below. |
| 3. Q1.5 wording | Accept | The split doesn't change the model, it changes how well we choose k and estimate performance. This is a more accurate answer to the question. |


### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/land_mines.csv")
X = df[["voltage", "height", "soil"]]
y = df["mine_type"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.50, random_state=65)

scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)

model = KNeighborsClassifier(n_neighbors=2).fit(X_train_s, y_train)
y_hat = model.predict(X_valid_s)

table = pd.crosstab(y_valid, y_hat, rownames=["actual"], colnames=["predicted"])
print(table, "\n")

diag = np.diag(table.to_numpy())
print("Correct / actual in class (how many real mines of each type we catch):")
print((pd.Series(diag, index=table.index) / table.sum(axis=1)).round(2), "\n")
print("Correct / predicted as class (how often a prediction of that type is right):")
print((pd.Series(diag, index=table.columns) / table.sum(axis=0)).round(2))



predicted   1   2   3  4  5
actual                     
1          21   1   6  4  2
2           0  33   2  3  0
3           5   5  10  6  4
4          13   5  10  9  1
5          10   1   8  7  3 

Correct / actual in class (how many real mines of each type we catch):
actual
1    0.62
2    0.87
3    0.33
4    0.24
5    0.10
dtype: float64 

Correct / predicted as class (how often a prediction of that type is right):
predicted
1    0.43
2    0.73
3    0.28
4    0.31
5    0.30
dtype: float64


### Written answers (Phase 2)

**Q1.5 (revised).** Splitting the data lets us fit the model on the training set and compare values of $k$ on data it has not seen. This does not improve the model itself. It improves our *choice* of $k$ and gives an honest estimate of how the model will do on new data, since training error alone would always favor the most flexible model (overfitting).

**Q2.3 – Selecting k.** The three features are on different scales, so I rescaled them to [0, 1] with `MinMaxScaler`, fitting the scaler on the training data only so the validation data does not influence the model. I then fit a k-NN classifier for every $k$ from 1 to 50 and counted the number of incorrect predictions on the validation set. The fewest mistakes occurred at $k = 2$, so I used that value.

**Q2.4 – Accuracy.** The model correctly classifies 76 of the 169 validation observations (45.0%), well above the 20% expected from guessing among five balanced classes but far from reliable. Performance varies by type. Mine type 2 is the best, with 33 of 38 caught (87%). Type 1 is next, with 21 of 34 (62%). Types 3, 4 and 5 are the worst, with only 10/30, 9/38 and 3/29 correct. These types are frequently confused with each other and with type 1.

**Q2.5 – Practical use.** Accuracy for a type can look good while the predictions are still often wrong. The model catches 21 of the 34 real type 1 mines (62%), but it labels 49 mines as type 1, so a "type 1" prediction is correct only 43% of the time. Type 4 and type 5 predictions are right only about 30% of the time, and type 5 mines are almost never identified correctly (3 of 29). Only a prediction of type 2 is reasonably trustworthy (73%). I would therefore not use this model to decide how a mine is handled. It could be used as a rough first screen, but every prediction other than type 2 should be confirmed by a human expert or additional measurements, and when the type is uncertain the mine should be handled with the most cautious procedure rather than the one for the predicted type.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

I think overall, AI was pretty good for correcting and helping me better understand the content. I had to prompt it twice actually, because the first time it was suggesting way too many things that were completely outside the scope of the assignment and stuff that I had never seen so I had to tell it to give its improvements based on the assignment's requirements. After reprompting through it was a lot better and clearer. I verfied the final solution by running its code improvement and seeing if it matches up and making sure that its output actually makes sense. I think a lot of the improvement it made was just how my comments were not fleshed out enough, such as the 2.5 correction. One thing that AI did that was nice is that it ran the code I wrote itself first to see the numbers and then evaluated after that to see if my answers aligned with the assingment requirements. I think seeing it do that made me more certain that the AI wasn't just completely hallucinating everything, which is can sometimes do.